# Laboratorio 2 — Evaluación de la calidad de los datos

**Asignatura:** Analítica de Datos  
**Caso:** Andina Retail

## Objetivo

Evaluar la calidad inicial del dataset e identificar problemas que puedan afectar el análisis y la interpretación de los resultados.

En este laboratorio **no corregiremos los problemas encontrados**.

Seguiremos tres acciones:

**detectar → cuantificar → interpretar**

## 1. Carga del dataset
Comenzamos nuevamente desde el archivo original para garantizar que el análisis sea reproducible.


In [1]:
import pandas as pd
df = pd.read_csv("/content/ventas.csv")

In [2]:
df.head()

,fecha,id_cliente,producto,categoria,ciudad,unidades,precio_unitario,descuento,canal,ventas
0,2025-11-03,13375,Guantes gym,Deportes,Bogota,4,48100.0,0.05,En linea,182780.0
1,2025-06-05,19441,Licuadora,Hogar,Barranquilla,5,81800.0,0.00,Tienda,409000.0
2,2025-06-02,10603,Tapete yoga,Deportes,Bogota,2,75200.0,0.10,En linea,135360.0
3,2025-11-15,10698,Bicicleta estatica,Deportes,Bogota,3,42300.0,0.00,Tienda,126900.0
4,2025-08-29,15539,Tenis urbanos,Moda,Cali,1,94500.0,0.05,Tienda,89775.0


In [3]:
df.shape

(4820, 10)

## 2. Perfil estadístico de las variables numéricas

Antes de buscar errores específicos, examinaremos las magnitudes y la distribución básica de las variables numéricas.

Preguntas:

- ¿Los valores mínimos y máximos parecen plausibles?
- ¿Existen diferencias importantes entre media y mediana?
- ¿Todas las variables tienen la misma cantidad de observaciones?

In [4]:
df.describe()

,id_cliente,unidades,precio_unitario,descuento,ventas
count,4820.000000,4820.000000,4.820000e+03,4702.000000,4.796000e+03
mean,14991.975934,2.665353,1.255276e+05,0.068099,2.732060e+05
std,2886.987778,7.267589,1.661873e+05,0.078400,3.777610e+05
min,10000.000000,1.000000,9.100000e+03,0.000000,1.220000e+04
25%,12537.000000,1.000000,4.717500e+04,0.000000,9.040000e+04
50%,14956.500000,3.000000,7.010000e+04,0.050000,1.662250e+05
75%,17532.250000,3.000000,1.168000e+05,0.100000,3.125600e+05
max,19995.000000,500.000000,3.200000e+06,0.300000,1.245000e+07


## 3. Valores faltantes

Ahora determinaremos qué variables contienen información faltante y qué proporción del dataset representan esos valores.

In [5]:
df.isnull().sum()

,0
fecha,0
id_cliente,0
producto,0
categoria,0
ciudad,0
unidades,0
precio_unitario,0
descuento,118
canal,0
ventas,24


In [6]:
faltantes = pd.DataFrame({
    "cantidad": df.isnull().sum(),
    "porcentaje": (df.isnull().sum() / len(df) * 100).round(2)
})
faltantes

,cantidad,porcentaje
fecha,0,0.00
id_cliente,0,0.00
producto,0,0.00
categoria,0,0.00
ciudad,0,0.00
unidades,0,0.00
precio_unitario,0,0.00
descuento,118,2.45
canal,0,0.00
ventas,24,0.50


## 4. Registros duplicados

Un registro duplicado es una fila cuyos valores coinciden con los de otra fila.

Primero determinaremos cuántos duplicados exactos existen.

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df[df.duplicated(keep=False)]

,fecha,id_cliente,producto,categoria,ciudad,unidades,precio_unitario,descuento,canal,ventas


## 5. Consistencia de variables categóricas

Las variables categóricas pueden presentar inconsistencias de escritura que generan categorías artificialmente diferentes.

Comenzaremos examinando `ciudad`.

In [9]:
df["ciudad"].value_counts()

,count
ciudad,
Medellin,1402
Bogota,1310
Cali,1080
Barranquilla,900
MEDELLIN,87
bogota,41


### Revisión de otras variables categóricas

Verificaremos si existen problemas similares en categoría, canal y producto.

In [10]:
df["categoria"].value_counts()

,count
categoria,
Hogar,1543
Moda,1484
Deportes,980
Tecnologia,813


In [11]:
df["producto"].value_counts()

,count
producto,
Juego de ollas,347
Chaqueta impermeable,333
Lampara LED,308
Licuadora,306
Jean slim,294
Set de sabanas,291
Aspiradora,291
Tenis urbanos,290
Camiseta basica,289


## 6. Cardinalidad de las variables categóricas

La cardinalidad indica cuántos valores distintos aparecen en una variable.

In [12]:
columnas_categoricas = [
    "producto",
    "categoria",
    "ciudad",
    "canal"
]

for columna in columnas_categoricas:
    print(f"{columna}: {df[columna].nunique()} valores únicos")

producto: 20 valores únicos
categoria: 4 valores únicos
ciudad: 6 valores únicos
canal: 2 valores únicos


# 9. Registro de hallazgos

A partir del perfilado realizado, se identificaron los siguientes hallazgos:

| Variable / aspecto | Hallazgo | Evidencia | Riesgo para el análisis |
|---|---|---|---|
| `descuento` | Valores faltantes | 118 valores faltantes (2.45 %) | Los análisis relacionados con descuentos podrían utilizar información incompleta |
| `ventas` | Valores faltantes | 24 valores faltantes (0.50 %) | Puede afectar cálculos de ventas totales, medias y comparaciones entre grupos |
| `ciudad` | Categorías inconsistentes | `Bogota` / `bogota` y `Medellin` / `MEDELLIN` | Una misma ciudad puede ser contabilizada como categorías diferentes |
| Registros | No se detectaron duplicados exactos | `df.duplicated().sum()` = 0 | No existe evidencia de doble contabilización causada por filas exactamente duplicadas |
| `categoria` | No se observaron inconsistencias evidentes | 4 valores: Hogar, Moda, Deportes y Tecnologia | No requiere tratamiento evidente por codificación |
| `canal` | No se observaron inconsistencias evidentes | 2 valores únicos | No requiere tratamiento evidente por codificación |

## Interpretación

El dataset presenta una calidad inicial razonable, pero **no está todavía listo para el análisis**.

Los principales problemas detectados son:

1. presencia de valores faltantes en `descuento` y `ventas`;
2. inconsistencia en la codificación de algunas ciudades.

No todos los controles realizados revelaron problemas. La ausencia de duplicados exactos y la consistencia observada en `categoria` y `canal` también constituyen resultados del proceso de evaluación de calidad.

# 10. Cierre del Laboratorio 2

## ¿Qué podemos afirmar ahora?

El proceso de evaluación permitió establecer que:

- el dataset contiene **4.820 registros y 10 variables**;
- `descuento` presenta **118 valores faltantes (2.45 %)**;
- `ventas` presenta **24 valores faltantes (0.50 %)**;
- no se detectaron **registros exactamente duplicados**;
- `ciudad` presenta inconsistencias de escritura:
  - `Bogota` / `bogota`;
  - `Medellin` / `MEDELLIN`;
- `categoria` presenta **4 categorías** sin inconsistencias evidentes;
- `canal` presenta **2 valores únicos** sin inconsistencias evidentes;
- se identificaron **20 productos diferentes**.

## Lo que todavía NO hemos hecho

Hasta este punto únicamente hemos:

**detectado → cuantificado → interpretado**

No hemos eliminado, reemplazado ni transformado ningún dato.

---

## Pregunta para continuar

**¿Qué tratamiento debemos aplicar a cada problema sin alterar injustificadamente la información original?**

Esta pregunta conduce a la siguiente etapa de CRISP-DM:

### Data Preparation

y al:

### Laboratorio 3 — Preparación de los datos